In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.neighbors import KNeighborsRegressor

def f(X):
    X = np.atleast_2d(X)
    return 0.5 * X[:, 0] + np.sin(X[:, 1]) - 0.3 * X[:, 2] ** 2

In [29]:
rng = np.random.default_rng(seed=42)

n_train = 400
n_test = 1000
p = 30

cov = np.full((p, p), 0.8) + np.diag(np.full(p, 0.2))

X1_train = rng.standard_normal((n_train, p))
X1_test = rng.standard_normal((n_test, p))
X2_train = rng.multivariate_normal(np.zeros(p), cov, n_train)
X2_test = rng.multivariate_normal(np.zeros(p), cov, n_test)

y1_train = f(X1_train) + rng.normal(0, 0.1, n_train)
y1_test = f(X1_test) + rng.normal(0, 0.1, n_test)
y2_train = f(X2_train) + rng.normal(0, 0.1, n_train)
y2_test = f(X2_test) + rng.normal(0, 0.1, n_test)

In [30]:
def fit_knn(X_train, y_train, X_test, y_test):
    knn = KNeighborsRegressor(n_neighbors=5, metric="euclidean")
    knn.fit(X_train, y_train)
    return np.mean((y_test - knn.predict(X_test))**2)

In [ ]:
def fit_lasso(X_train, y_train, X_test, y_test, seed=0):
    pipe = make_pipeline(StandardScaler(), Lasso(fit_intercept=True, max_iter=100000))
    grid = {"lasso__alpha": np.logspace(-4, 0, 41)}
    cv = GridSearchCV(
        pipe, grid,
        cv = KFold(n_splits=10, shuffle=True, random_statex=seed),
        scoring = "neg_mean_squared_error"
    )
    cv.fit(X_train, y_train)

    lam = cv.best_params_["lasso__alpha"]
    test_mse = np.mean((y_test - cv.predict(X_test)) ** 2)
    n_nonzero = int(np.sum(cv.best_estimator_.named_steps["lasso"].coef_ != 0))
    return lam, test_mse, n_nonzero

In [35]:
knn1_mse = fit_knn(X1_train, y1_train, X1_test, y1_test)
knn2_mse = fit_knn(X2_train, y2_train, X2_test, y2_test)

lambda1, lasso1_mse, nz1 = fit_lasso(X1_train, y1_train, X1_test, y1_test)
lambda2, lasso2_mse, nz2 = fit_lasso(X2_train, y2_train, X2_test, y2_test)

results_table = pd.DataFrame({
    "Design": ["Independent", "Correlated (ρ = 0.8)"],
    "5NN test MSE": [knn1_mse, knn2_mse],
    "Lasso test MSE": [lasso1_mse, lasso2_mse],
    "Selected λ": [lambda1, lambda2],
    "Nonzero slopes": [nz1, nz2],
})

In [36]:
print(results_table.style.hide(axis="index").format({
    "5NN test MSE": "{:.4f}",
    "Lasso test MSE": "{:.4f}",
    "Selected λ": "{:.4g}",
}).to_typst())

#table(
  columns: 6,
  [], [Design], [5NN test MSE], [Lasso test MSE], [Selected λ], [Nonzero slopes],

  [], [Independent], [0.5548], [0.2698], [0.03162], [7],
  [], [Correlated (ρ = 0.8)], [0.1770], [0.3103], [0.01259], [15],
)
